In [47]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from scipy.stats import chi2_contingency

import src.fct_data
import importlib
importlib.reload(src.fct_data) 

from src.fct_data import raw_data_cleaning


## Extraction des données



Traitement pour les 3 années disponibles sur le `gitlab` 

In [42]:
#on récupère un df avec métadonnées + texte nettoyé des stopwords
meta_et_texts=raw_data_cleaning("data/archelect_search.zip","data/text_files")

/home/onyxia/work/NLP_3A/src/fct_data.py:12: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")


Traitement de data/text_files/1981/legislatives.zip
Traitement de data/text_files/1993/legislatives.zip
Traitement de data/text_files/1993/presidentielle.zip
Traitement de data/text_files/1988/legislatives.zip
Nombre total de documents extraits : 12746


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-t

## [NEW] Identifcation des thèmes pour l'élection 1981

### Première approche : BoW + IDF 

Pour montrer que ça ne marche pas bien pcq on rate les thèmes

In [70]:
#1981
texts_1981 = meta_et_texts[meta_et_texts['annee'] == 1981]
print(f"Nombre de documents 1981 : {len(texts_1981)}")

#on selectionne les partis présentants le plus de candidats
parti_counts = texts_1981['suppleant-soutien'].value_counts()
parti_counts = parti_counts.drop("non mentionné", errors='ignore')
top_partis = parti_counts.head(6).index.tolist()

rows=[]

#matrice BoW IDF
vectorizer = TfidfVectorizer(
    stop_words=None,   
    max_features=30    # garder les 30 mots les plus fréquents
)

for parti in top_partis:
    # Filtrer les textes de ce parti
    texts_parti = texts_1981[texts_1981['suppleant-soutien'] == parti]['texte_clean']
    
    idf_matrix = vectorizer.fit_transform(texts_parti)

    idf_df = pd.DataFrame(idf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
    top_parti = idf_df.mean(axis=0).sort_values(ascending=False).index.tolist()
      # Ajouter au tableau
    rows.append({
        "parti": parti,
        "top_words": top_parti
    })


top_words_df = pd.DataFrame(rows)
top_words_df['top_words'] = top_words_df['top_words'].apply(lambda x: ", ".join(x))
top_words_df

Nombre de documents 1981 : 3121


,parti,top_words
0,Parti socialiste,"france, socialiste, majorité, plus, politique,..."
1,Parti communiste français,"gauche, majorité, changement, communistes, com..."
2,Lutte ouvrière,"gauche, mitterrand, faut, travailleurs, plus, ..."
3,Parti socialiste unifié,"psu, majorité, gauche, cest, plus, politique, ..."
4,Rassemblement pour la République,"france, plus, politique, fonds, po, majorité, ..."
5,Rassemblement pour la République;Union pour la...,"plus, majorité, france, juin, candidat, politi..."


## Représentation textuelle

In [35]:
model = SentenceTransformer('camembert-base')
embeddings = model.encode(meta_et_texts['texte_clean'].tolist(), batch_size=16, show_progress_bar=True)
print(f"Embeddings : {embeddings.shape}")

No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 574.05it/s, Materializing param=pooler.dense.weight]                               
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:  11%|█         | 88/797 [05:40<45:40,  3.87s/it]


KeyboardInterrupt: 

## Classification par thème

In [10]:
umap_embeddings = umap.UMAP(
    n_neighbors=15, min_dist=0.0, n_components=5, random_state=42
).fit_transform(embeddings)
print(f"UMAP embeddings shape: {umap_embeddings.shape}")

/opt/python/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embeddings shape: (3182, 5)


In [11]:
print("Clustering avec HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=10)
meta_et_texts['theme_cluster'] = clusterer.fit_predict(umap_embeddings)
print(f"Clusters trouvés : {meta_et_texts['theme_cluster'].nunique()} (le -1 correspond aux outliers)")


Clustering avec HDBSCAN...
Clusters trouvés : 17 (le -1 correspond aux outliers)


## Croisement avec les professions des candidats

On fait un peu de visualisation

In [12]:
plt.figure(figsize=(12,6))
sns.countplot(x='theme_cluster', hue='profession', data=meta_et_texts)
plt.title("Répartition des thèmes abordés par métier")
plt.xlabel("Thème")
plt.ylabel("Nombre de professions de foi")
plt.legend(title="Métier")
plt.show()

ValueError: Could not interpret value `profession` for `hue`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## Analyses statistiques diverses

In [ ]:
contingency_table = pd.crosstab(meta_et_texts['profession'], meta_et_texts['theme_cluster'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi2 = {chi2:.2f}, p-value = {p:.4f}")